# A1 – Calidad de datos

Primero miramos los datos tal como vinieron (`cargar_crudo`) y al final comprobamos que `cargar_limpio` aplica lo que decidimos en `DECISIONES.md`.

1. Perfil de cada tabla
2. Chequeos entre tablas
3. Las tres anomalías más importantes
4. Métricas de calidad por dataset
5. ¿Los datos cuentan la misma historia que el enunciado?
6. Modelo relacional
7. Verificación de `cargar_limpio`

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
from src.carga import cargar_crudo, cargar_limpio, RAW, FECHAS

d = cargar_crudo()
cli = d["clientes"]
trx = d["transacciones"]
prd = d["productos"]
dev = d["devoluciones"]
crm = d["interacciones_crm"]
cam = d["campanas_marketing"]

for nombre, tabla in d.items():
    print(f"{nombre:20s} {tabla.shape[0]:6d} filas  {tabla.shape[1]:2d} columnas")

## 1. Perfil de cada tabla

Leemos todo como texto para que ningún valor raro se convierta en NaN sin que nos demos cuenta. Buscamos celdas vacías, nulos "disfrazados" (`null`, `N/A`, `No especificado`…), duplicados, rangos y fechas que no se puedan leer.

In [ ]:
NULOS_DISFRAZADOS = ["nan", "null", "none", "n/a", "na", "-", "?", "no especificado", "desconocido"]

for nombre in d:
    t = pd.read_csv(RAW / f"{nombre}.csv", dtype=str, keep_default_na=False)
    id_col = t.columns[0]
    print(f"\n=== {nombre}: {len(t)} filas")
    print(f"duplicados exactos: {t.duplicated().sum()}  |  {id_col} repetidos: {t[id_col].duplicated().sum()}")

    for col in t.columns:
        s = t[col].str.strip()
        vacios = (s == "").sum()
        disfrazados = s.str.lower().isin(NULOS_DISFRAZADOS).sum()
        linea = f"  {col:22s} vacíos={vacios:4d}  disfrazados={disfrazados:4d}  únicos={s.nunique():6d}"

        num = pd.to_numeric(s, errors="coerce")
        if num.notna().mean() > 0.9:
            linea += f"  | min={num.min():g}  mediana={num.median():g}  max={num.max():g}"
        elif col in FECHAS.get(nombre, []):
            f = pd.to_datetime(s, errors="coerce", format="%Y-%m-%d")
            linea += f"  | {f.min().date()} a {f.max().date()}  (no se pueden leer: {f.isna().sum() - vacios})"
        elif s.nunique() <= 10:
            linea += f"  | {s.value_counts().to_dict()}"
        print(linea)

**Qué vemos:** no hay celdas vacías ni duplicados en ninguna tabla. El único nulo disfrazado es `genero = "No especificado"` (433 clientes), que tratamos como dato faltante en la completitud. Todas las fechas van del 2023-01-01 al 2024-12-31 (24 meses). Hay 318 productos, no los 320 que dice el enunciado.

## 2. Chequeos entre tablas

Incluye los que pide el enunciado (devoluciones sin transacción de origen, fechas que no tienen sentido) y otros de aritmética de precios.

In [ ]:
def mostrar(texto, valor):
    print(f"{texto:62s} {valor}")

print("--- Claves huérfanas")
mostrar("transacciones con cliente inexistente", (~trx.cliente_id.isin(cli.cliente_id)).sum())
mostrar("transacciones con producto inexistente", (~trx.producto_id.isin(prd.producto_id)).sum())
mostrar("devoluciones sin transacción de origen", (~dev.transaccion_id.isin(trx.transaccion_id)).sum())
for nombre, t in [("devoluciones", dev), ("crm", crm), ("campañas", cam)]:
    mostrar(f"{nombre} con cliente inexistente", (~t.cliente_id.isin(cli.cliente_id)).sum())
mostrar("clientes sin ninguna transacción", (~cli.cliente_id.isin(trx.cliente_id)).sum())

print("\n--- Transacciones")
calculado = (trx.cantidad * trx.precio_unitario * (1 - trx.descuento)).round(2)
mostrar("monto_neto distinto de cantidad*precio*(1-descuento)", ((trx.monto_neto - calculado).abs() > 0.05).sum())
m = trx.merge(prd[["producto_id", "precio_lista", "costo", "categoria"]], on="producto_id")
mostrar("precio_unitario distinto del precio de lista", ((m.precio_unitario - m.precio_lista).abs() > 0.01).sum())
mostrar("venta por unidad por debajo del costo", (m.monto_neto / m.cantidad < m.costo).sum())
con_registro = trx.merge(cli[["cliente_id", "fecha_registro"]], on="cliente_id", how="left")
mostrar("compra anterior al registro del cliente", (con_registro.fecha < con_registro.fecha_registro).sum())

print("\n--- Productos")
margen_calc = ((prd.precio_lista - prd.costo) / prd.precio_lista * 100).round(1)
mostrar("margen_bruto_pct que no cuadra con costo y precio", ((margen_calc - prd.margen_bruto_pct).abs() > 0.15).sum())

print("\n--- Devoluciones")
dd = dev.merge(trx[["transaccion_id", "cliente_id", "fecha", "monto_neto"]],
               on="transaccion_id", how="left", suffixes=("", "_trx"))
mostrar("devolución antes de la compra", (dd.fecha_devolucion < dd.fecha).sum())
mostrar("se devolvió más de lo que se pagó", (dd.monto_devuelto > dd.monto_neto + 0.01).sum())
mostrar("el cliente de la devolución no es el de la compra", (dd.cliente_id != dd.cliente_id_trx).sum())
mostrar("días de la compra a la devolución (mediana / p95 / máx)",
        (dd.fecha_devolucion - dd.fecha).dt.days.quantile([0.5, 0.95, 1]).tolist())
mostrar("% de transacciones con devolución", round(len(dev) / len(trx) * 100, 2))

print("\n--- CRM")
q1, q3 = crm.tiempo_respuesta_hrs.quantile([0.25, 0.75])
umbral = q3 + 1.5 * (q3 - q1)
mostrar(f"tiempo de respuesta sobre Q3+1,5·IQR ({umbral:.1f} h)", (crm.tiempo_respuesta_hrs > umbral).sum())
mostrar("tiempo de respuesta de más de una semana", (crm.tiempo_respuesta_hrs > 168).sum())

print("\n--- Campañas")
mostrar("convertido=1 sin haber abierto", ((cam.convertido == 1) & (cam.abierto == 0)).sum())
mostrar("conversión global % / apertura %", f"{cam.convertido.mean():.2%} / {cam.abierto.mean():.2%}")

print("\n--- Segmento declarado vs comportamiento")
ultima = trx.groupby("cliente_id").fecha.max()
c = cli.set_index("cliente_id").join(ultima.rename("ultima"))
recientes = c.ultima >= trx.fecha.max() - pd.Timedelta(days=90)
mostrar("'Inactivo' con compras en los últimos 90 días", ((c.segmento == "Inactivo") & recientes).sum())

## 3. Las tres anomalías más importantes

1. ¿Las compras anteriores al registro se explican por el canal (gente que compraba en tienda antes de registrarse)?
2. ¿Qué hay detrás de las ventas bajo costo?
3. ¿Las conversiones de campaña tienen una compra que las respalde? ¿Hay campañas "convertidas" que se enviaron después de la compra?

In [ ]:
con_registro["antes"] = con_registro.fecha < con_registro.fecha_registro
print("1) % de compras antes del registro, por canal:")
print(con_registro.groupby("canal").antes.mean().round(3).to_string())

por_cliente = con_registro.groupby("cliente_id").agg(registro=("fecha_registro", "first"), primera=("fecha", "min"))
correlacion = por_cliente.registro.map(pd.Timestamp.toordinal).corr(por_cliente.primera.map(pd.Timestamp.toordinal))
print("   correlación entre fecha de registro y primera compra:", round(correlacion, 3))

In [ ]:
bajo_costo = m[m.monto_neto / m.cantidad < m.costo]
print("2) Ventas bajo costo:", len(bajo_costo))
print("   descuento aplicado (mín / mediana / máx):", bajo_costo.descuento.quantile([0, 0.5, 1]).round(3).tolist())
print("   por categoría:", bajo_costo.categoria.value_counts().to_dict())
print("   margen de lista de esas categorías:", prd.groupby("categoria").margen_bruto_pct.median().round(1).to_dict())

In [ ]:
convertidas = cam[cam.convertido == 1]
cruce = convertidas.merge(trx[["cliente_id", "fecha"]], on="cliente_id", how="left")
cruce["dias"] = (cruce.fecha - cruce.fecha_envio).dt.days

print(f"3) {len(convertidas)} campañas marcadas como convertidas")
for ventana in (7, 14, 30):
    con_compra = cruce[cruce.dias.between(0, ventana)].campana_id.unique()
    print(f"   con compra en los {ventana:2d} días siguientes: {convertidas.campana_id.isin(con_compra).sum()}")

despues = set(cruce[cruce.dias.between(0, 30)].campana_id)
antes = set(cruce[cruce.dias.between(-30, -1)].campana_id)
print("   solo hay una compra en los 30 días ANTERIORES al envío:", len(antes - despues))

**Conclusiones (detalle en `DECISIONES.md`):**

1. Las compras antes del registro pasan igual en los tres canales y la fecha de registro no tiene relación con la primera compra. Como la empresa tiene 12 años y todos los registros son de 2023–2024, lo interpretamos como la fecha de alta en la plataforma nueva, no como el inicio de la relación con el cliente. No usamos `fecha_registro` para medir antigüedad.
2. Las ventas bajo costo son casi todas de Electrónica, que tiene el margen de lista más bajo: con descuentos del 9% al 34,5% el margen se vuelve negativo. No es un error, es un hallazgo para A2.
3. Solo 179 de 797 conversiones tienen una compra en los 30 días siguientes, y en 102 casos la única compra cercana es **anterior** al envío (el caso que menciona el enunciado). El campo `convertido` sirve para comparar tasas, pero no para calcular cuánto dinero trajo una campaña.

## 4. Métricas de calidad por dataset

Definiciones que usamos:

- **Completitud**: % de celdas con un valor real (sin vacíos ni nulos disfrazados).
- **Unicidad**: % de filas cuyo ID no se repite.
- **Consistencia**: % de filas que cumplen todas las reglas de su tabla (debajo se ve cuántas incumple cada regla).
- **Oportunidad**: % de registros cuya fecha cae dentro de los 24 meses del estudio y cuya información está completa al corte (por ejemplo, las compras de diciembre todavía pueden recibir devoluciones).

In [ ]:
inicio, fin = pd.Timestamp("2023-01-01"), pd.Timestamp("2024-12-31")

def completitud(nombre):
    t = pd.read_csv(RAW / f"{nombre}.csv", dtype=str, keep_default_na=False)
    faltan = t.apply(lambda s: (s.str.strip() == "") | s.str.strip().str.lower().isin(NULOS_DISFRAZADOS))
    return 100 * (1 - faltan.values.mean())

ult_compra = trx.groupby("cliente_id").fecha.max()
cli_ult = cli.cliente_id.map(ult_compra)
precio_ok = trx.merge(prd[["producto_id", "precio_lista"]], on="producto_id", how="left").set_index(trx.index)

reglas = {
    "clientes": {
        "edad entre 18 y 100": cli.edad.between(18, 100),
        "género informado": cli.genero != "No especificado",
        "'Inactivo' sin compras en los últimos 90 días": ~((cli.segmento == "Inactivo") & (cli_ult >= fin - pd.Timedelta(days=90))),
    },
    "transacciones": {
        "cliente existe": trx.cliente_id.isin(cli.cliente_id),
        "producto existe": trx.producto_id.isin(prd.producto_id),
        "monto_neto cuadra": (trx.monto_neto - calculado).abs() <= 0.05,
        "precio = precio de lista": (precio_ok.precio_unitario - precio_ok.precio_lista).abs() <= 0.01,
        "compra posterior al registro": ~con_registro.set_index(trx.index).antes,
    },
    "productos": {
        "costo < precio de lista": prd.costo < prd.precio_lista,
        "margen_bruto_pct cuadra": (margen_calc - prd.margen_bruto_pct).abs() <= 0.15,
    },
    "devoluciones": {
        "transacción de origen existe": dev.transaccion_id.isin(trx.transaccion_id),
        "mismo cliente que la compra": (dd.cliente_id == dd.cliente_id_trx).values,
        "después de la compra": (dd.fecha_devolucion >= dd.fecha).values,
        "no devuelve más de lo pagado": (dd.monto_devuelto <= dd.monto_neto + 0.01).values,
    },
    "interacciones_crm": {
        "cliente existe": crm.cliente_id.isin(cli.cliente_id),
        "tiempo de respuesta ≥ 0": crm.tiempo_respuesta_hrs >= 0,
        "respuesta en menos de una semana": crm.tiempo_respuesta_hrs <= 168,
    },
    "campanas_marketing": {
        "cliente existe": cam.cliente_id.isin(cli.cliente_id),
        "convertida implica abierta": ~((cam.convertido == 1) & (cam.abierto == 0)),
        "conversión con compra en 30 días": ~((cam.convertido == 1) & ~cam.campana_id.isin(despues)),
    },
}

# Oportunidad: fecha dentro del período y, donde aplica, con tiempo suficiente para que el dato esté completo
p95_devolucion = int((dd.fecha_devolucion - dd.fecha).dt.days.quantile(0.95))
oportunidad = {
    "clientes": cli.fecha_registro.between(inicio, fin),
    "transacciones": trx.fecha <= fin - pd.Timedelta(days=p95_devolucion),
    "productos": pd.Series(True, index=prd.index),  # no tiene fecha
    "devoluciones": dev.fecha_devolucion.between(inicio, fin),
    "interacciones_crm": crm.fecha.between(inicio, fin),
    "campanas_marketing": cam.fecha_envio <= fin - pd.Timedelta(days=30),
}

filas = []
for nombre, t in d.items():
    ok = pd.concat({k: pd.Series(np.asarray(v), index=t.index) for k, v in reglas[nombre].items()}, axis=1)
    filas.append({
        "dataset": nombre,
        "filas": len(t),
        "completitud %": round(completitud(nombre), 2),
        "unicidad %": round(100 * (1 - t.iloc[:, 0].duplicated().mean()), 2),
        "consistencia %": round(100 * ok.all(axis=1).mean(), 2),
        "oportunidad %": round(100 * oportunidad[nombre].mean(), 2),
    })
    print(f"\n{nombre}")
    for regla, cumple in ok.items():
        print(f"   {regla:48s} incumplen: {(~cumple).sum()}")

calidad = pd.DataFrame(filas).set_index("dataset")
calidad

**Cómo leer la tabla:**

- La completitud y la unicidad están prácticamente al 100%: los datos vienen "limpios" en forma. Los problemas son de **significado**, no de formato.
- La consistencia de `transacciones` baja a ~50% por una sola regla (compra antes del registro), que explicamos arriba: no es un error de la transacción sino del significado de `fecha_registro`.
- La consistencia de `campanas_marketing` refleja las conversiones sin compra que las respalde.
- En oportunidad, marcamos como "incompletas al corte" las compras de los últimos 29 días (p95 del tiempo hasta una devolución) y las campañas del último mes (no alcanzó a pasar la ventana de conversión). En la práctica el efecto es pequeño (diciembre queda en 13,4% de devoluciones, en línea con el resto), pero lo tenemos presente al mirar tendencias de fin de período.

In [ ]:
# Comprobación de lo anterior: tasa de devolución por mes de compra
tasa_mes = trx.assign(devuelta=trx.transaccion_id.isin(dev.transaccion_id)).groupby(trx.fecha.dt.to_period("M")).devuelta.mean() * 100
print(f"p95 de días hasta la devolución: {p95_devolucion}")
print(tasa_mes.tail(4).round(1).to_string())
print(f"promedio del resto del período: {tasa_mes.iloc[:-2].mean():.1f}%")

## 5. ¿Los datos cuentan la misma historia que el enunciado?

El caso dice: ticket promedio −8% interanual, devoluciones subiendo al 14% el último trimestre y conversión de campañas cayendo del 22% al 11% en 18 meses. Lo comprobamos antes de construir el relato sobre esas cifras.

In [ ]:
por_anio = trx.groupby(trx.fecha.dt.year).monto_neto.mean()
print("Ticket promedio por año:", por_anio.round(2).to_dict(),
      f"-> variación {100 * (por_anio[2024] / por_anio[2023] - 1):.1f}%")

trx_q = trx.fecha.dt.to_period("Q")
tasa_dev = dev.merge(trx[["transaccion_id"]].assign(q=trx_q), on="transaccion_id").groupby("q").size() / trx.groupby(trx_q).size() * 100
conversion = cam.groupby(cam.fecha_envio.dt.to_period("Q")).convertido.mean() * 100
pd.DataFrame({"tasa devolución %": tasa_dev.round(1), "conversión %": conversion.round(1)})

**Resultado:** los datos no confirman dos de las tres cifras del enunciado.

| Métrica | Enunciado | Datos |
|---|---|---|
| Ticket promedio | −8% interanual | −1,9% (351 → 344) |
| Devoluciones | suben al 14% | estables entre 13,0% y 13,8% todo el período |
| Conversión | 22% → 11% | 11,3% → 5,6%: se cae a la mitad, pero desde otro nivel |

Además aparece un tercer canal de venta (`app_movil`, 17% de las transacciones) que el enunciado no menciona, y `transacciones` trae `monto_neto`. Esto es el punto de partida del relato: el problema real y grave es el de marketing; las devoluciones son un costo alto pero estable, no una tendencia nueva.

## 6. Modelo relacional

El diagrama está en `docs/modelo.md` (esquema estrella). Resumen:

- **Hecho principal:** `transacciones` (una fila = un producto comprado en una fecha por un cliente). Las medidas son cantidad, monto_neto y margen.
- **Hecho dependiente:** `devoluciones`, que cuelga de una transacción (1 a 0..1).
- **Hechos de eventos del cliente:** `campanas_marketing` e `interacciones_crm`. No tienen llave hacia una transacción, solo hacia el cliente; por eso la relación campaña → compra hay que inferirla por fecha.
- **Dimensiones:** `clientes`, `productos` y una dimensión fecha (común a todas las tablas de hechos).

La cardinalidad la comprobamos aquí:

In [ ]:
print("devoluciones por transacción (máx):", dev.transaccion_id.value_counts().max())
print("¿cada devolución apunta a una sola transacción?", dev.devolucion_id.is_unique)
print("transacciones por cliente (mediana / máx):", trx.cliente_id.value_counts().median(), trx.cliente_id.value_counts().max())
print("campañas por cliente (mediana / máx):", cam.cliente_id.value_counts().median(), cam.cliente_id.value_counts().max())
print("interacciones por cliente (mediana / máx):", crm.cliente_id.value_counts().median(), crm.cliente_id.value_counts().max())

## 7. Verificación de `cargar_limpio`

Comprobamos que no se pierde ninguna fila y que las marcas coinciden con lo que contamos arriba.

In [ ]:
limpio = cargar_limpio(recalcular=True)

for nombre in d:
    assert len(limpio[nombre]) == len(d[nombre]), nombre

t, c, k = limpio["transacciones"], limpio["clientes"], limpio["campanas_marketing"]
print("ventas bajo costo:", t.bajo_costo.sum())
print("outliers de monto por categoría:", t.outlier_cat.sum())
print("clientes sin compras:", c.sin_compras.sum())
print("conversiones respaldadas:", k.conversion_respaldada.sum(), "de", (k.convertido == 1).sum())
print("CRM con respuesta atípica:", limpio["interacciones_crm"].respuesta_atipica.sum())
print(f"margen bruto total: {t.margen_bruto.sum():,.0f} sobre ventas de {t.monto_neto.sum():,.0f} ({t.margen_bruto.sum() / t.monto_neto.sum():.1%})")
pd.crosstab(c.segmento, c.segmento_rfm, margins=True)

El cruce del final muestra que el segmento declarado y el comportamiento coinciden a grandes rasgos, pero no del todo (por ejemplo, muchos "Esporádico" y "Nuevo" llevan más de 180 días sin comprar). En A2 y A3 usamos el segmento declarado, que es el que pide el enunciado, y `segmento_rfm` para contrastar.